# NF-UNSW-NB15 — TTL Ablasyonu (Run All)

Doğrulama çalışması, bu korpusta **üç özelliğin tek başına** sınıfları ayırdığını
gösterdi:

| Özellik | Tek eşikle doğruluk |
|---|---|
| `MAX_TTL` | 0.9948 |
| `MIN_TTL` | 0.9948 |
| `MIN_IP_PKT_LEN` | 0.9916 |

Ancak `MAX_TTL` tek başına çıkarıldığında performans hiç düşmedi — çünkü
`MIN_TTL` onun neredeyse birebir kopyası. Bu nedenle **tek özellik ablasyonu
bilgi vermez**; grubun birlikte çıkarılması gerekir.

Bu notebook o deneyi yapar:

1. Makaledeki **grup-farkındalı bölmeyi** (host çifti, seed 42) yeniden kurar.
2. Dört yapılandırmada Random Forest eğitir ve **tutulan-set** (held-out)
   başarımını ölçer — örnek-içi değil.
3. TTL grubu çıkarıldıktan sonra **başka bir özelliğin devralıp devralmadığını**
   kontrol eder.
4. Makaleye yapıştırılacak LaTeX cümlesini üretir.

**Süre:** 4 × RF eğitimi, Colab'ın 2 çekirdeğinde yaklaşık **5–8 dakika**. GPU
gerekmez.

## Hücre 1 — Veri setini bulma

In [ ]:
# ============================================================
# HUCRE 1 — VERIYI OTOMATIK BULMA
# ============================================================
import os, glob, json, datetime
import numpy as np, pandas as pd

# Ana notebook'un kullandigi klasor: Mamba_BTD/datasets/NF-UNSW-NB15
# Bu yol ONCELIKLE denenir; bulunamazsa genel arama devreye girer.
MANUEL_CSV = ""      # <-- tam yolu BILIYORSANIZ buraya yazin, arama atlanir
TERCIH_EDILEN = [
    "/content/drive/MyDrive/Mamba_BTD/datasets/NF-UNSW-NB15/NF-UNSW-NB15-v2.csv",
    "/content/drive/My Drive/Mamba_BTD/datasets/NF-UNSW-NB15/NF-UNSW-NB15-v2.csv",
]
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d_%H%M%S")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print(f"[BILGI] drive.mount calistirilmadi ({type(e).__name__}).")

def bul():
    if MANUEL_CSV:
        return MANUEL_CSV, [MANUEL_CSV]
    # 1) Tercih edilen yol var mi?
    for t in TERCIH_EDILEN:
        if os.path.exists(t):
            return t, [t]
    # 2) TUM kopyalari topla -- Drive'da birden fazla surum olabilir
    kokler = [r for r in ["/content/drive/MyDrive", "/content/drive/My Drive",
                          "/content/drive/Shareddrives", "/content", os.getcwd()]
              if os.path.isdir(r)]
    adaylar = []
    for r in kokler:
        for dsn in ("NF-UNSW-NB15*.csv", "NF_UNSW_NB15*.csv"):
            adaylar += glob.glob(os.path.join(r, "**", dsn), recursive=True)
    adaylar = sorted(set(adaylar))
    if not adaylar:
        return None, []
    # 3) 'Mamba_BTD/datasets' iceren yolu TERCIH ET (ana notebook'un yapisi)
    oncelikli = [a for a in adaylar
                 if "mamba_btd" in a.lower() and "datasets" in a.lower()]
    return (oncelikli[0] if oncelikli else adaylar[0]), adaylar

CSV, ADAYLAR = bul()
if len(ADAYLAR) > 1:
    print(f"\n[DIKKAT] Drive'da {len(ADAYLAR)} adet NF-UNSW-NB15 dosyasi bulundu:")
    for a in ADAYLAR:
        print(f"    {'>>> SECILDI' if a == CSV else '           '}  {a}")
    print("    Yanlis kopya secildiyse MANUEL_CSV degiskenine dogru yolu yazin.")
assert CSV and os.path.exists(CSV), (
    "NF-UNSW-NB15 CSV bulunamadi. MANUEL_CSV degiskenine tam yolu yazin.")
print(f"[OK] veri: {CSV}")

# Cikti klasoru: CSV'den geriye dogru proje kokunu bul
BASE = os.path.dirname(os.path.dirname(os.path.dirname(CSV)))
OUT_DIR = os.path.join(BASE, "verification")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[OK] cikti: {OUT_DIR}")

## Hücre 2 — Ön işleme ve grup-farkındalı bölme

Makaledeki boru hattıyla **aynı** işlemler: kimlik sütunları düşürülür, kategorik
sütunlar tam sayıya kodlanır, Inf/NaN satırları atılır, bölme host çifti üzerinde
`GroupShuffleSplit(test_size=0.2, random_state=42)` ile yapılır.

In [ ]:
# ============================================================
# HUCRE 2 — ON ISLEME + GRUP-FARKINDALI BOLME
# ============================================================
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(CSV, low_memory=False)
df.columns = [c.strip() for c in df.columns]
lab = "Label" if "Label" in df.columns else "label"
atk = "Attack" if "Attack" in df.columns else lab

groups = df["IPV4_SRC_ADDR"].astype(str) + "__" + df["IPV4_DST_ADDR"].astype(str)
y_all = df[lab].astype(int)
X_all = df.drop(columns=[c for c in ["IPV4_SRC_ADDR", "IPV4_DST_ADDR", lab, atk]
                         if c in df.columns], errors="ignore")
for c in X_all.select_dtypes(include=["object", "string"]).columns:
    X_all[c] = pd.factorize(X_all[c].astype(str))[0]
X_all = X_all.replace([np.inf, -np.inf], np.nan)
keep = X_all.notna().all(axis=1)
X, y, g = X_all[keep].astype(np.float32), y_all[keep].values, groups[keep]
del df, X_all, y_all

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2,
                                random_state=42).split(X, groups=g))
BEKLENEN_TOPLAM = 139264      # makalede raporlanan NF-UNSW-NB15 kayit sayisi
BEKLENEN_POZ    = 0.50        # makaledeki pozitif oran (~0.50, dengeli alt-kume)

print(f"ozellik sayisi : {X.shape[1]}")
print(f"train / test   : {len(tr):,} / {len(te):,}")
print(f"benzersiz grup : {g.nunique():,}")
print(f"ortak grup     : {len(set(g.iloc[tr]) & set(g.iloc[te]))}  (0 olmali)")
print(f"pozitif oran   : train={y[tr].mean():.4f}  test={y[te].mean():.4f}")

TTL_GRUBU = [c for c in ["MAX_TTL", "MIN_TTL", "MIN_IP_PKT_LEN"] if c in X.columns]
print(f"\nablasyona girecek ozellikler: {TTL_GRUBU}")

# ---- KORUMA 1: DOGRU KORPUS MU? ----
toplam = len(X)
poz = float(y.mean())
print("\n" + "=" * 62)
if abs(toplam - BEKLENEN_TOPLAM) > 0.02 * BEKLENEN_TOPLAM:
    print("*** UYARI: BU, MAKALEDE KULLANILAN DOSYA DEGIL ***")
    print(f"    bulunan  : {toplam:,} kayit, pozitif oran {poz:.4f}")
    print(f"    beklenen : {BEKLENEN_TOPLAM:,} kayit, pozitif oran ~{BEKLENEN_POZ:.2f}")
    print("    Drive'da NF-UNSW-NB15-v2.csv'nin BIRDEN FAZLA kopyasi olabilir")
    print("    (orn. tam surum ve calismada kullanilan alt-kume).")
    print("    MANUEL_CSV degiskenine ANA NOTEBOOK'un kullandigi yolu yazip")
    print("    bu notebook'u YENIDEN calistirin; aksi halde sonuclar makaleyle")
    print("    KIYASLANAMAZ.")
    KORPUS_UYUMLU = False
else:
    print(f"[OK] Korpus makaleyle uyumlu: {toplam:,} kayit, pozitif oran {poz:.4f}")
    KORPUS_UYUMLU = True
print("=" * 62)

## Hücre 3 — Ablasyon

Dört yapılandırma, hepsi aynı bölme ve aynı hiperparametrelerle. **Tutulan-set**
doğruluğu, makro-F1 ve AUC raporlanır.

In [ ]:
# ============================================================
# HUCRE 3 — ABLASYON (4 x RandomForest, ~5-8 dk)
# ============================================================
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

YAPILANDIRMALAR = {
    "tum ozellikler":              [],
    "MAX_TTL haric":               ["MAX_TTL"],
    "iki TTL de haric":            ["MAX_TTL", "MIN_TTL"],
    "TTL + MIN_IP_PKT_LEN haric":  ["MAX_TTL", "MIN_TTL", "MIN_IP_PKT_LEN"],
}
YAPILANDIRMALAR = {k: [c for c in v if c in X.columns]
                   for k, v in YAPILANDIRMALAR.items()}

ABL = {}
print(f"{'yapilandirma':30s} {'ozellik':>7s} {'dogruluk':>9s} "
      f"{'makro-F1':>9s} {'AUC':>7s} {'sure':>7s}")
print("-" * 74)
for ad, drop in YAPILANDIRMALAR.items():
    cols = [c for c in X.columns if c not in drop]
    t0 = time.time()
    rf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42,
                                class_weight="balanced")
    rf.fit(X.iloc[tr][cols], y[tr])
    p = rf.predict(X.iloc[te][cols])
    pr = rf.predict_proba(X.iloc[te][cols])[:, 1]
    acc = accuracy_score(y[te], p)
    f1 = f1_score(y[te], p, average="macro")
    try:
        auc = roc_auc_score(y[te], pr)
    except Exception:
        auc = float("nan")
    dt = time.time() - t0
    ABL[ad] = dict(dropped=drop, n_features=len(cols), accuracy=float(acc),
                   macro_f1=float(f1), auc=float(auc))
    print(f"{ad:30s} {len(cols):7d} {acc:9.4f} {f1:9.4f} {auc:7.4f} {dt:6.0f}s")

taban = ABL["tum ozellikler"]["accuracy"]
son = ABL[list(YAPILANDIRMALAR)[-1]]["accuracy"]
print(f"\ntoplam dusus: {taban:.4f} -> {son:.4f}  (delta = {taban - son:+.4f})")

## Hücre 4 — TTL'siz dünyada başka bir artefakt var mı?

TTL grubu çıkarıldıktan sonra kalan özellikler arasında **tek eşikle** yüksek
doğruluk veren başka bir sütun varsa, sorun tek bir gruba özgü değil demektir.
Bu, makale için TTL bulgusundan bile önemli olabilir.

In [ ]:
# ============================================================
# HUCRE 4 — KALAN OZELLIKLERDE TEK-ESIK TARAMASI
# ============================================================
def en_iyi_tek_esik(x, yy, n=256):
    qs = np.unique(np.quantile(x, np.linspace(0.001, 0.999, n)))
    best = 0.0
    for t in qs:
        p = (x > t).astype(int)
        best = max(best, accuracy_score(yy, p), accuracy_score(yy, 1 - p))
    return float(best)

kalan = [c for c in X.columns if c not in TTL_GRUBU]
rf_k = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42,
                              class_weight="balanced").fit(X.iloc[tr][kalan], y[tr])
imp = pd.Series(rf_k.feature_importances_, index=kalan).sort_values(ascending=False)

# KORUMA 2: dengesiz bir test setinde tek-esik dogrulugu COGUNLUK SINIFI
# taban cizgisinin ALTINA inemez. Taban cizgisi raporlanmadan bu sayilar
# yorumlanamaz: 0.96 degeri, benign orani 0.96 ise HICBIR SEY ifade etmez.
TABAN = max(y[te].mean(), 1 - y[te].mean())
print(f"cogunluk-sinifi taban cizgisi (test): {TABAN:.4f}")
print("bir ozellik ancak bu tabani BELIRGIN olcude asiyorsa artefakt adayidir\n")
print("TTL grubu cikarildiktan sonra, en onemli 10 ozellik icin")
print("TEK esikle ulasilan en yuksek dogruluk (test setinde):\n")
tek = {}
for c in imp.head(10).index:
    tek[c] = en_iyi_tek_esik(X.iloc[te][c].values, y[te])
    # tabana gore FAZLALIK: mutlak deger degil, tabani ne kadar astigi onemli
    fazla = tek[c] - TABAN
    bayrak = "   <-- YENI ARTEFAKT ADAYI" if fazla > 0.02 else ""
    print(f"  onem={imp[c]:6.4f}  tek-esik={tek[c]:.4f}  {c}{bayrak}")

en_yuksek = max(tek, key=tek.get)
print(f"\nen yuksek: {tek[en_yuksek]:.4f} ({en_yuksek}), "
      f"tabanin {tek[en_yuksek] - TABAN:+.4f} uzerinde")
ABL["_kalan_tek_esik"] = {k: float(v) for k, v in tek.items()}
ABL["_kalan_en_yuksek"] = [en_yuksek, float(tek[en_yuksek])]
ABL["_taban_cizgisi"] = float(TABAN)
ABL["_korpus_uyumlu"] = bool(KORPUS_UYUMLU)
ABL["_n_kayit"] = int(len(X))

## Hücre 5 — Özet ve **yapıştırılacak LaTeX**

Üretilen cümle, Tartışma Bölüm 5.7'nin sonuna eklenecektir.

In [ ]:
# ============================================================
# HUCRE 5 — OZET + LATEX
# ============================================================
def tex(n):
    for a, b in [("_", "\\_"), ("%", "\\%"), ("&", "\\&"), ("#", "\\#")]:
        n = n.replace(a, b)
    return n

taban = ABL["tum ozellikler"]
tek_ttl = ABL["MAX_TTL haric"]
iki_ttl = ABL["iki TTL de haric"]
uc = ABL["TTL + MIN_IP_PKT_LEN haric"]
kalan_ad, kalan_acc = ABL["_kalan_en_yuksek"]

def _fiil(onceki, simdiki):
    """Deger gercekten dustu mu? Cumleyi olcume gore kur."""
    d = onceki - simdiki
    if d < 0.005:
        return "leaves it at"
    return "reduces it to"

cumle = (
    f"Removing \\texttt{{MAX\\_TTL}} alone "
    f"{'leaves held-out accuracy essentially unchanged at' if taban['accuracy'] - tek_ttl['accuracy'] < 0.005 else 'reduces held-out accuracy from ' + format(taban['accuracy'], '.4f') + ' to'} "
    f"{tek_ttl['accuracy']:.4f}, as the redundancy predicts; removing both "
    f"time-to-live fields {_fiil(tek_ttl['accuracy'], iki_ttl['accuracy'])} "
    f"{iki_ttl['accuracy']:.4f}, and removing the three near-deterministic "
    f"features together {_fiil(iki_ttl['accuracy'], uc['accuracy'])} "
    f"{uc['accuracy']:.4f} "
    f"(macro-$F_{{1}}$ {uc['macro_f1']:.4f}, AUC {uc['auc']:.4f}). "
)
if kalan_acc - ABL["_taban_cizgisi"] > 0.02:
    cumle += (f"A residual artefact remains: with those features withheld, "
              f"\\texttt{{{tex(kalan_ad)}}} still separates the classes at "
              f"{kalan_acc:.4f} accuracy under a single threshold, so the "
              f"corpus is not rendered artefact-free by removing the "
              f"time-to-live group alone")
else:
    cumle += (f"No remaining feature exceeds {kalan_acc:.4f} accuracy under a "
              f"single threshold, so the near-determinism is confined to the "
              f"time-to-live group rather than being pervasive")

print("=" * 70)
if not ABL.get("_korpus_uyumlu", True):
    print("METIN URETILMEDI")
    print("=" * 70)
    print(f"\nBu calisma {ABL['_n_kayit']:,} kayitlik bir dosyada yapildi; makale")
    print(f"{BEKLENEN_TOPLAM:,} kayit raporluyor. Sonuclar KIYASLANABILIR DEGIL")
    print("ve makaleye konulmamalidir. MANUEL_CSV'yi ana notebook'un kullandigi")
    print("dosyaya ayarlayip yeniden calistirin.")
    raise SystemExit
print("MAKALEYE YAPISTIRILACAK METIN")
print("=" * 70)
print("\nDOSYA : BTD_discussion.tex")
print("YER   : Bolum 5.7, 'a meaningful ablation would have to remove the")
print("        group.' cumlesinin HEMEN ARDINA yeni cumle olarak ekleyin.\n")
print(cumle + ".")

kayit = os.path.join(OUT_DIR, f"ttl_ablation_{RUN_ID}.json")
with open(kayit, "w") as f:
    json.dump(dict(run_id=RUN_ID, csv=CSV, n_train=int(len(tr)), n_test=int(len(te)),
                   ablation=ABL, latex=cumle + "."), f, indent=2, default=str)
print(f"\n[KAYDEDILDI] {kayit}")